# Find orthologues between lab and singing mice

In [1]:
# load packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# snRNAseq
import anndata as ad
import scanpy as sc

# for calc MAD
from scipy.stats import median_abs_deviation

Singing mouse trinity transcripts, purported Malat1
- ST_new trinity
- TRINITY_DN1295_c1_g1
- TRINITY_DN329_c3_g1
`seqkit grep -r -p "^(TRINITY_DN1295_c1_g1_i|TRINITY_DN329_c3_g1_i)" /Volumes/S2D1/cindy/forEmily/new_st_all_trinity_out_dir.Trinity.fasta -o st_trinity_malat1_seqs.fasta`



Lab mouse trinity
- TRINITY_DN11831_c0_g1
- TRINITY_DN2630_c0_g1
- TRINITY_DN839_c4_g1
`seqkit grep -r -p "^(TRINITY_DN11831_c0_g1_i|TRINITY_DN2630_c0_g1_i|TRINITY_DN839_c4_g1_i)" /Volumes/S2D1/cindy/forEmily/trinity_out_dir.Trinity.fasta -o mm_trinity_malat1_seqs.fasta`

Extract transcripts from trinity:
`seqtk subseq Trinity.fasta ids.txt > extracted_transcripts.fasta`


Found no matches b/w lab mouse transcripts and malat1_mm_rna, but found single match for sining mouse seq: **TRINITY_DN1295_c1_g1_i1**

In [1]:
# Align mm_genome_malat1 and st_trinity_malt1

from Bio import Align
from Bio import SeqIO


In [2]:

# Load your two sequences
mm_gnome_malat1 = SeqIO.read("gene_seqs/mmus_malt1_NR_002847.3.fa", "fasta").seq
st_trin_malat1 = SeqIO.read("gene_seqs/st_malat1_trinity_match.fa", "fasta").seq


In [4]:
# Set up the aligner for global (end-to-end) alignment
aligner = Align.PairwiseAligner()
aligner.mode = "local"  # this is what makes it end-to-end (Needleman-Wunsch style)

# Scoring parameters — adjust based on how divergent you expect these to be
# commented out to use default parameters
# aligner.match_score = 2
# aligner.mismatch_score = -1
# aligner.open_gap_score = -10
# aligner.extend_gap_score = -0.5

# Run the alignment
alignments = aligner.align(mm_gnome_malat1, st_trin_malat1)

# Get the best (first) alignment
best_alignment = alignments[0]
print(best_alignment)
print("Score:", best_alignment.score)

target            0 CAGGCATTCAG-GCAGCGAGAGCAGAGCAGCGTAG-AGC-AGCACAGCTGAG-CTCGTGA
                  0 |.||..|||..-||...|.|-|||-||||.....|-|||-||.|||...|||-||..|||
query           105 CGGGTCTTCCCCGCCATGCG-GCA-AGCATTCAGGCAGCGAGGACACAGGAGGCTGTTGA

target           56 GGCAGGAGACTCAGCCCGAGGAAATCGCAGATAAGTTTTTAATTAAAAAGATTGAGCAGT
                 60 ||||||||||.||||..|||||||||||||.|||||||.||.||||||||||||||-|.|
query           163 GGCAGGAGACGCAGCTTGAGGAAATCGCAGCTAAGTTTCTACTTAAAAAGATTGAG-ATT

target          116 AAAAAGAATTAGAACTCTAAACTTAAGCT---AA-TAGAGTAGCTTATCGAAAT-A---T
                120 ||.||-|||||.||||||||||||||.||---||-|||||||||||.|.||.||-|---|
query           222 AAGAA-AATTAAAACTCTAAACTTAAACTGTTAAATAGAGTAGCTTGTGGAGATGACGCT

target          168 TACT-T-AG-TCTT-AA--TAA-T-CTAA-G-AAGA---TCTTAAGAGATAACATGAAGG
                180 ||.|-|-|.-|.||-||--|||-|-|||.-|-||.|---|.||||||||.||.|||||||
query           281 TAGTATTAAATTTTTAAAGTAACTTCTAGCGTAAAAAGGTTTTAAGAGAAAATATGAAGG

target          215 CTTA

In [5]:
# Count matches for percent identity
aligned_seq1 = str(best_alignment[0])
aligned_seq2 = str(best_alignment[1])

matches = sum(a == b for a, b in zip(aligned_seq1, aligned_seq2) if a != '-' and b != '-')
aligned_length = len(aligned_seq1)
percent_identity = (matches / aligned_length) * 100

print(f"Percent identity: {percent_identity:.2f}%")
print(f"Alignment length: {aligned_length}")

Percent identity: 79.82%
Alignment length: 7249


In [ ]:
# visualize